## Section A: The Mental Model

In [15]:
import pandas as pd

# Load the Excel file
df = pd.read_excel('cAlwUp3sSzSjtoyrfBXL_saas.xlsx')

# Export to CSV
df.to_csv('SAAS.CSV', index=False)

In [16]:
df

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
0,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
1,A-1002,Beta LLC,2023-02-14 00:00:00,US,5.0,450,pro,2024-03-02,False
2,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
3,A-1003,Cirrus,1675209600,GB,0.0,0,pro,2024-02-28,False
4,A-1004,Delta Inc,2023-03-20 00:00:00,United States,12.0,1188,pro,2024-03-01,No
5,A-1005,Echo,2023-04-01 00:00:00,US,8.0,-99,pro,2024-03-03,False
6,A-1006,Foxtrot,2023-05-11 00:00:00,us,3.0,270,PRO,2024-03-02,False
7,A-1007,Gamma,2023-06-30 00:00:00,DE,15.0,1800,enterprise,2023-06-15,True
8,A-1008,Helio,2023-07-22 00:00:00,US,NaN,540,pro,2024-03-04,False
9,A-1009,Iris,2023-08-08 00:00:00,US,10.0,900,pro,NaT,True


#### Q1. Predict the alignment [10marks]
Without running anything, predict the exact output of the code below, including every index label and every value, including anything that is not a number. Then run it and reconcile.

In [17]:
# the actual output
a= pd.Series({'jan':100, 'feb':200, 'mar':300})
b= pd.Series({'feb':10, 'mar':20, 'apr':40})
result= a+b
print(result)

apr      NaN
feb    210.0
jan      NaN
mar    320.0
dtype: float64


* Pandas Series Creation: The dictionaries create two Pandas Series where the keys (jan, feb, mar, apr) serve as index labels and the numbers serve as values.
* Index Alignment: When adding two Series together with +, Pandas automatically aligns the data by matching index labels rather than positional order.
* Matching Keys: Labels present in both Series (feb and mar) have their values added together ($200+10=210.0$ and $300+20=320.0$).
* Missing Keys & Data Types: Labels found in only one Series (jan and apr) result in NaN (Not a Number), which converts the whole result's data type to floating-point (float64).

#### Q2. Series or DataFrame? [8marks]
For each expression below, state whether it returns a Series or a DataFrame, and give one concrete reason the difference would matter in later code (name a specific operation that behaves differently)

In [18]:
#df['mrr'] 
#df[['mrr']]
#df.loc[:, 'mrr']
#df.loc[:,['mrr', 'seats']]

| Expression | Returned Type | Specific Operation | Simple Consequence / Practical Output Difference |
| :--- | :--- | :--- | :--- |
| `df['mrr']` | **Series** | `.sum()` | Returns a **single scalar number** (`13019`) that you can directly use in basic math operations. |
| `df[['mrr']]` | **DataFrame** | `.sum()` | Returns a **labeled Series output** (`mrr 13019`) that preserves the column structure. |
| `df.loc[:, 'mrr']` | **Series** | Addition (`+ 10`) | Adds 10 to each element and outputs a **1D Series** of values. |
| `df.loc[:, ['mrr', 'seats']]` | **DataFrame** | Multi-column selection | Preserves a **2D table structure** holding multiple columns, which a Series cannot do. |

### Key Reminder
- Single Bracket / Single Label (df['mrr'] or .loc[:, 'mrr']): Extracts raw data (a Series/1D list). Best when you just want values or single numbers.

- Double Bracket / List Label (df[['mrr']] or .loc[:, ['mrr', 'seats']]): Keeps the table structure (a DataFrame/2D table). Best when you want to keep data neatly formatted in columns.

## Section B: Column Slicing

#### Q3.Four routes to the same columns [12marks]
Write four different one-line expressions that each select exactly the columns company, seats, and mrr, no more and no fewer. Use a genuinely different mechanism each time (by name, by .loc label range, by .iloc position, and by .filter or select_dtypes). Then answer: two of your four routes are fragile if a colleague reorders or renames columns upstream. Which two, and why? Which single route would you put in production code, and why?

In [19]:
# By Name (Double Bracket Selection)
df[['company', 'seats', 'mrr']]
# By .loc Label Range
df.loc[:, 'company':'mrr']
# By .iloc Position
df.iloc[:, 0:3]
# By .filter
df.filter(items=['company', 'seats', 'mrr'])

,company,seats,mrr
0,Acme Corp,25.0,2500
1,Beta LLC,5.0,450
2,Acme Corp,25.0,2500
3,Cirrus,0.0,0
4,Delta Inc,12.0,1188
5,Echo,8.0,-99
6,Foxtrot,3.0,270
7,Gamma,15.0,1800
8,Helio,NaN,540
9,Iris,10.0,900


#### Fragility Analysis
The two fragile routes are `.loc` label range `(df.loc[:, 'company':'mrr'])` and `.iloc` position `(df.iloc[:, 0:3])`.

- Why `iloc` position is fragile: `.iloc` relies entirely on fixed zero-based column indices. If an upstream step inserts, removes, or reorders any column, position `0:3` will silently pull entirely wrong columns (or fail with an `IndexError`), corrupting downstream calculations without warning.

- Why ``.loc`` label range is fragile: ``.loc`` slicing includes every column between the start label (``'company'``) and end label (``'mrr'``). If a colleague reorders columns or inserts a new column between ``company`` and ``mrr``, the slice will dynamically expand or shrink, pulling unexpected columns into your subset.

#### Production Choice & Justification
**Production Choice:** Direct selection by explicit name list — df[['company', 'seats', 'mrr']]

**Justification:**

- **Order Agnostic:** It extracts the exact named columns regardless of where they sit in the DataFrame or if upstream steps reorder columns.

- **Fail-Fast Validation:** If a required column (company, seats, or mrr) is accidentally dropped or renamed upstream, direct bracket indexing immediately raises a KeyError. This instantly halts pipeline execution, preventing bad or incomplete data from silently propagating into production models or databases. (Note: .filter() silently drops missing columns rather than raising an error, making it less safe for strict production schemas).

- **Readability:**' It is self-documenting and explicitly states data requirements to future maintainers.

#### Q4.The inclusive/exclusive trap
Suppose the columns are in the order listed above. Predict, before running, how many columns each of these returns and name them:
* df.loc[:, 'company':'mrr']
* df.iloc[:, 1:5]
  
They look like they should return the samething. Do they? Explain the rule that governs each, in your own words.

In [20]:
#df.loc[:, 'company':'mrr']
df.iloc[:, 1:5]

,company,signup_ts,country,seats
0,Acme Corp,2023-01-05 09:00:00,US,25.0
1,Beta LLC,2023-02-14 00:00:00,US,5.0
2,Acme Corp,2023-01-05 09:00:00,US,25.0
3,Cirrus,1675209600,GB,0.0
4,Delta Inc,2023-03-20 00:00:00,United States,12.0
5,Echo,2023-04-01 00:00:00,US,8.0
6,Foxtrot,2023-05-11 00:00:00,us,3.0
7,Gamma,2023-06-30 00:00:00,DE,15.0
8,Helio,2023-07-22 00:00:00,US,NaN
9,Iris,2023-08-08 00:00:00,US,10.0


#### Written Predictions
The DataFrame columns start with [account_id,	company,	signup_ts,	country,	seats,	mrr,	plan,	last_active,	churned] at positional indices 0, 1, 2, ...:

``df.loc[:, 'company':'mrr']``

- Column Count: 5 columns
- Predicted Names: ['company', 'signup_ts', 'country', 'seats', 'mrr']

``df.iloc[:, 1:5]``

- Column Count: 4 columns
- Predicted Names: ['company', 'signup_ts', 'country', 'seats']

In [21]:
# Actual Column Lists
df.loc[:, 'company':'mrr']  # [company, signup_ts, country, seats, mrr] (5 columns)
df.iloc[:, 1:5]  # [company, signup_ts, country, seats] (4 columns)

,company,signup_ts,country,seats
0,Acme Corp,2023-01-05 09:00:00,US,25.0
1,Beta LLC,2023-02-14 00:00:00,US,5.0
2,Acme Corp,2023-01-05 09:00:00,US,25.0
3,Cirrus,1675209600,GB,0.0
4,Delta Inc,2023-03-20 00:00:00,United States,12.0
5,Echo,2023-04-01 00:00:00,US,8.0
6,Foxtrot,2023-05-11 00:00:00,us,3.0
7,Gamma,2023-06-30 00:00:00,DE,15.0
8,Helio,2023-07-22 00:00:00,US,NaN
9,Iris,2023-08-08 00:00:00,US,10.0


#### Do They Return the Same Thing?
No, they do not return the same thing. They differ in two fundamental ways:

**Different Start Columns:** df.loc[:, 'company':'mrr'] includes the starting column selected, 'company' (position 0). df.iloc[:, 1:5] starts at index 1, which skips 'account_id' entirely and begins at 'company'.

**Different Column Counts:** df.loc returns 5 columns, whereas df.iloc returns 4 columns.

#### Rule Explanation (The Inclusive/Exclusive Trap)
**.loc (Label-Based Slicing — Inclusive/Inclusive):**
When you slice strings/labels with .loc, both endpoints are inclusive. Pandas includes the start label, the stop label, and every column in between. It behaves this way because with arbitrary labels, there is no natural "next" label to serve as an exclusive boundary.

**.iloc (Position-Based Slicing — Inclusive/Exclusive):**
When you slice integer indices with .iloc, it strictly follows standard Python sequence slicing rules (start:stop). The start index is inclusive, but the stop index is exclusive. 1:5 selects elements at positions 1, 2, 3, and 4, stopping right before position 5.

## Section C: Diagnosis and Cleaning 

#### Q5.The count is the question [12marks] 
Open the data and list everydata-quality problem you can find. For each, state (a) what the problem is, (b) which of the four questions it belongs to (what is one row/what should each columnbe/what is missing or lying/what shape do I need), and (c) whether you would fix it, and why. At least one "problem" in this dataset is not actually dirt. Identify it and defend leaving it alone. Full marks require you to distinguish real dirt from legitimate data.

| # | Column(s) Affected | Row(s) | Diagnostic Question | Problem Description | Action & Justification |
| :-: | :--- | :-: | :--- | :--- | :--- |
| **1** | All Columns | Row 2 | **What is one row?** | **Exact Duplicate Row:** Row 2 (`A-1001`) is a 100% identical copy of Row 0 across all attributes. | **FIX:** Drop Row 2 (`df.drop_duplicates()`). Duplicate records inflate account counts and overall revenue. |
| **2** | `account_id`, `seats`, `mrr`, `last_active` | Rows 1 & 12 | **What is one row?** | **Primary Key Collision / Version Drift:** `A-1002` (Beta LLC) appears twice with conflicting states (Row 1: 5 seats/$450 MRR; Row 12: 6 seats/$540 MRR on a later date). | **FIX:** Keep only the most recent snapshot (Row 12, `last_active = 2024-03-06`) and drop Row 1. Account tables must maintain 1 row = 1 account. |
| **3** | `signup_ts` | Row 3 | **What should each column be?** | **Inconsistent Date Encoding:** `signup_ts` contains a raw Unix Epoch integer (`1675209600`) while all other rows use ISO standard strings (`YYYY-MM-DD HH:MM:SS`). | **FIX:** Convert using `pd.to_datetime(unit='s')` so the column becomes unified datetime objects (yields `2023-02-01 00:00:00`). |
| **4** | `country` | Rows 4 & 6 | **What should each column be?** | **Inconsistent Categorical Values:** Uses `'United States'`, `'us'` (lowercase), and `'US'` (uppercase) to represent the same entity. | **FIX:** Standardize all values to ISO-2 uppercase codes (`'US'`). Inconsistencies break grouping and aggregation. |
| **5** | `plan` | Row 6 | **What should each column be?** | **Inconsistent Casing:** Row 6 uses `'PRO'` in uppercase, whereas other rows use lowercase `'pro'`. | **FIX:** Lowercase all plan strings (`df['plan'].str.lower()`). |
| **6** | `churned` | Row 4 | **What should each column be?** | **Mixed Boolean Types:** Uses string `'No'` alongside boolean strings `'False'` and `'True'`. | **FIX:** Convert the entire column to strict boolean data types (`True`/`False`). |
| **7** | `mrr` | Row 5 | **What is missing or lying?** | **Impossible Negative MRR:** `A-1005` (Echo) has `mrr = -99`. Monthly Recurring Revenue cannot be negative for an active, non-churned account. | **FIX:** Investigate or correct to positive (e.g., $99 or standard 8-seat Pro rate of $720). Negative values distort total MRR calculations. |
| **8** | `signup_ts`, `last_active` | Row 7 | **What is missing or lying?** | **Temporal Contradiction:** `last_active` (`2023-06-15`) occurs *before* `signup_ts` (`2023-06-30`). An account cannot be active before creation. | **FIX:** Nullify or correct `last_active`. Invalid temporal logic breaks customer retention and lifetime duration metrics. |
| **9** | `seats` | Row 8 | **What is missing or lying?** | **Missing Value (`NaN`):** `A-1008` (Helio) has `NaN` for `seats`, but holds an MRR of $540 on the Pro plan ($90/seat). | **FIX:** Impute `seats = 6` ($540 / $90 per seat). Domain rules allow exact recovery of missing values. |
| **10** | `seats` | Row 13 | **What is missing or lying?** | **Fat-Finger Outlier / Typo:** `A-1012` (Lima) lists `1,000,000.0` seats for only 90 MRR on the Pro plan. | **FIX:** Correct `seats` to `1.0`. A million seats for $90 is an obvious data entry error ($0.00009/seat) that severely skews mean calculations. |
| **11** | `last_active` | Row 14 | **What is missing or lying?** | **Sentinel / Future Placeholder Date:** `last_active` is set to `2099-01-01`. | **FIX:** Replace `2099-01-01` with `NaT` (or current system date). Hardcoded futuristic dates corrupt date arithmetic (e.g., recency/tenure). |
| **12** | `company` | Row 15 | **What is missing or lying?** | **Missing Text Attribute (`NaN`):** `company` is missing for `A-1014` | **DO NOT DROP ROW.** Fill missing name with `'Unknown'`. Dropping paying accounts distorts financial summaries. |

#### Distinguishing Real Dirt from Legitimate Data
**The False Positive ("Not Dirt"):** Zero MRR Enterprise Account

Row 10 (A-1010 - Juno): mrr = 0 with seats = 20.0 on the enterprise plan.

**Defense of Leaving It Alone:**
While an automated anomaly detector might flag $0 MRR with 20 seats as "missing revenue dirt," this represents legitimate B2B SaaS business reality:

- **Free Trials & Proof-of-Concepts (POCs):** Enterprise prospects are regularly granted multi-seat trial access prior to contract execution.

- **Freemium / Executive Tier / Partner Accounts:** Companies frequently issue complimentary enterprise seats to partners or key stakeholders.

- **Grace Periods / Deferred Billing:** Enterprise accounts often have seat provisioning active prior to contract start dates or invoicing cycles.

**Why fixing/altering it would corrupt the analysis:**

- Overwriting 0 MRR with an estimated value (e.g., 20 seats × $100 = $2,000) would falsely inflate actual revenue figures.

- Dropping the row would underestimate active seat counts and skew sales conversion metrics (trial-to-paid pipeline conversion).

- Conclusion: This row should be retained exactly as-is in financial and operational reporting.

#### Q6. The duplicate that isn't [12 marks]

Two account IDs appear more than once. A classmate proposes a single line: df =df.drop_duplicates().Predict exactly what this line does to each repeated account, then run it and check. Explain why drop_duplicates()is the wrong tool for atleast one of the repeats. Then write the code you would actually use, and state the one piece of business information you would need before committing to it.



Submit: Prediction, actual result, the flaw you found, your replacement code, and the missing information you named

In [22]:
df_corp=df.drop_duplicates()
df

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
0,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
1,A-1002,Beta LLC,2023-02-14 00:00:00,US,5.0,450,pro,2024-03-02,False
2,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
3,A-1003,Cirrus,1675209600,GB,0.0,0,pro,2024-02-28,False
4,A-1004,Delta Inc,2023-03-20 00:00:00,United States,12.0,1188,pro,2024-03-01,No
5,A-1005,Echo,2023-04-01 00:00:00,US,8.0,-99,pro,2024-03-03,False
6,A-1006,Foxtrot,2023-05-11 00:00:00,us,3.0,270,PRO,2024-03-02,False
7,A-1007,Gamma,2023-06-30 00:00:00,DE,15.0,1800,enterprise,2023-06-15,True
8,A-1008,Helio,2023-07-22 00:00:00,US,NaN,540,pro,2024-03-04,False
9,A-1009,Iris,2023-08-08 00:00:00,US,10.0,900,pro,NaT,True


#### 1. Prediction
Without parameters, df.drop_duplicates() checks for full-row equality (where every single column value across two rows is identical).

- A-1001 (Acme Corp): Rows 0 and 2 are 100% identical copies across all 9 columns. df.drop_duplicates() will drop Row 2 and keep Row 0.

- A-1002 (Beta LLC): Row 1 and Row 12 have identical account_id, company, signup_ts, country, and plan values, but different values for seats (5 vs. 6), mrr ($450 vs. $540), and last_active (2024-03-02 vs. 2024-03-06). Because they are not 100% identical across all columns, df.drop_duplicates() will fail to remove either row, leaving both rows in the DataFrame.

#### 2. Actual Result
Running df.drop_duplicates() yields the following dataframe state:

- Dataset Shape: Reduced from (17, 9) to (16, 9).

- Account A-1001: Collapsed to 1 row (Row 0 preserved, Row 2 dropped).

- Account A-1002: Remained as 2 rows (Row 1 and Row 12 both retained).

#### 3. The Flaw Found
df.drop_duplicates() is the wrong tool for A-1002 because it assumes duplicate data is purely structural System Dirt (identical repeated rows).

For A-1002, we are looking at a Primary Key Collision / Version Drift (Temporal Update). The account upgraded from 5 seats ($450 MRR) to 6 seats ($540 MRR) between March 2nd and March 6th, 2024.

Because default drop_duplicates() evaluates the entire row rather than the entity identifier (account_id), it leaves stale snapshot rows in the dataset. If account_id is intended to be a unique primary key, drop_duplicates() silently fails to enforce uniqueness.
  

In [23]:
# 1. Parse dates safely to handle temporal ordering
df['last_active_dt'] = pd.to_datetime(df['last_active'], errors='coerce')

# 2. Sort by account_id and last_active date to place the latest record last
# 3. Drop duplicates based specifically on the Primary Key ('account_id')
df_cleaned = (
    df.sort_values(by=['account_id', 'last_active_dt'], ascending=[True, True])
      .drop_duplicates(subset=['account_id'], keep='last')
      .drop(columns=['last_active_dt'])
      .reset_index(drop=True)
)

df_cleaned

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
0,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
1,A-1002,Beta LLC,2023-02-14 00:00:00,US,6.0,540,pro,2024-03-06,False
2,A-1003,Cirrus,1675209600,GB,0.0,0,pro,2024-02-28,False
3,A-1004,Delta Inc,2023-03-20 00:00:00,United States,12.0,1188,pro,2024-03-01,No
4,A-1005,Echo,2023-04-01 00:00:00,US,8.0,-99,pro,2024-03-03,False
5,A-1006,Foxtrot,2023-05-11 00:00:00,us,3.0,270,PRO,2024-03-02,False
6,A-1007,Gamma,2023-06-30 00:00:00,DE,15.0,1800,enterprise,2023-06-15,True
7,A-1008,Helio,2023-07-22 00:00:00,US,NaN,540,pro,2024-03-04,False
8,A-1009,Iris,2023-08-08 00:00:00,US,10.0,900,pro,NaT,True
9,A-1010,Juno,2023-09-01 00:00:00,US,20.0,0,enterprise,2024-03-01,False


#### 5. Missing Business Information
The Key Business Question:

Is SAAS.CSV intended to be a Current-State Dimension Table (1 row per unique customer entity) or an Event-Log / Transaction History Table (multiple rows tracking historical account changes over time)?

- If Dimension Table (Current State): Keeping only the latest record (keep='last') is correct, as Row 12 represents Beta LLC's current active state ($540 MRR).

- If Event-Log Table (Historical): Neither row should be dropped; instead, a structural transformation (e.g., adding snapshot_date or version_id) is required to avoid primary key violations while preserving audit history.

#### Q7. One column, several diseases [14marks]
Compute the implied revenue per seat (mrr divided by seats, as numbers). Most accounts cluster around one value. Using that baseline, at least four rows look wrong ford if ferent reasons. 
For each suspicious row, give your single most likely explanation and the fix it implies. Crucially: for at least one row, there are two competing explanations the data alone cannot distinguish. Identify that row and explain what extra evidence would settle it.

In [24]:
df['mrr_per_seat'] = df['mrr'] / df['seats']
print(df[['account_id', 'company', 'plan', 'seats', 'mrr', 'mrr_per_seat']])

   account_id    company        plan      seats   mrr  mrr_per_seat
0      A-1001  Acme Corp  enterprise       25.0  2500     100.00000
1      A-1002   Beta LLC         pro        5.0   450      90.00000
2      A-1001  Acme Corp  enterprise       25.0  2500     100.00000
3      A-1003     Cirrus         pro        0.0     0           NaN
4      A-1004  Delta Inc         pro       12.0  1188      99.00000
5      A-1005       Echo         pro        8.0   -99     -12.37500
6      A-1006    Foxtrot         PRO        3.0   270      90.00000
7      A-1007      Gamma  enterprise       15.0  1800     120.00000
8      A-1008      Helio         pro        NaN   540           NaN
9      A-1009       Iris         pro       10.0   900      90.00000
10     A-1010       Juno  enterprise       20.0     0       0.00000
11     A-1011       Kilo         pro        7.0   630      90.00000
12     A-1002   Beta LLC         pro        6.0   540      90.00000
13     A-1012       Lima         pro  1000000.0 

#### 1. Implied Revenue Per Seat Computation
Implied revenue per seat is calculated as mrr / seats.

2. Baseline and Per-Row Anomaly Diagnostics
   
* Established Baseline: Standard pro plan accounts cluster strictly at $90.00 / seat. $enterprise contracts baseline between $100.00 – $120.00 / seat.

Row 3 (A-1003 - Cirrus): NaN ($0 / 0 seats)

* Explanation: A zero-division error caused by an inactive or shell account registered with 0 seats and $0 MRR.

* Fix: Handle zero-division explicitly in Pandas using np.where(df['seats'] > 0, df['mrr'] / df['seats'], 0.0) to safely output $0.00.

Row 4 (A-1004 - Delta Inc): $99.00 / seat

* Explanation: Tax/VAT inclusion in MRR. At 12 seats on the Pro plan, expected net MRR is $1,080 (12 × $90). Adding a standard 10% sales tax yields $1,188 total ($99/seat). Gross collected revenue was logged instead of net recurring revenue.

* Fix: Strip out the 10% tax component ($1,188 / 1.10) $to record net MRR as $1,080 ($90/seat).

Row 5 (A-1005 - Echo): -$12.38 / seat
* Explanation: Accounting sign flip. A credit adjustment or refund was logged directly into the recurring revenue field for an active, non-churned account.

* Fix: Recalculate MRR from valid seat volume (8 seats × $90 = $720 MRR).

Row 8 (A-1008 - Helio): NaN (Missing Seats)
* Explanation: Missing value (NaN) in the seats column despite a valid $540 MRR on the Pro plan.

* Fix: Impute seats deterministically using the Pro unit rate ($540 / $90 = 6 seats).

3. The Row with Two Irreconcilable Readings
Target Row: Row 13 (A-1012 - Lima) — seats = 1,000,000.0, mrr = 90 ($0.00009 / seat).

The tabular data alone cannot distinguish between two equally valid root causes:

**i** Explanation A: Seat Column Typo (Fat-Finger)

* Premise: The customer purchased 1 seat at the standard Pro rate ($90/month), but extra zeros were keyed into the seat field.

* Implied Fix: Set seats = 1.0 and keep mrr = 90.

**ii** Explanation B: Unit Rate Entered as Total MRR

* Premise: Lima is a massive 1,000,000-seat enterprise deal, but the data operator entered the per-seat unit rate ($90) $into total MRR instead of the full contract value ($90,000,000).

* Implied Fix: Keep seats = 1,000,000.0 and update mrr = 90,000,000.

Extra Evidence Required to Settle
* Stripe / Billing Payment Logs: To confirm the actual monthly charge processed ($90 vs. $90,000,000).

* Identity Provider / Okta Logs: To count the actual provisioned active directory accounts associated with tenant A-1012.

* CRM / Executed Contract Document: To verify agreed seat quantities and total contract value in Salesforce.

#### Q8. Parsing is not validating [10 marks] 
The date columns contain more than one kind of problem. Some can be caught by parsing; at least two cannot, because the values are valid dates that are nonetheless impossible. Write code that surfaces both kinds. Then explain, in your own words, the difference between a parsing problem and a validation problem, using a specific example of each from this dataset.

In [25]:
# 1. SURFACING PARSING PROBLEMS (Syntax / Representation Failures)
# Standard pandas parsing assumes string ISO format ("YYYY-MM-DD")
# We identify values that fail standard datetime string conversion:
parsed_signup_std = pd.to_datetime(df['signup_ts'], errors='coerce')
parsing_failures = df[df['signup_ts'].notna() & parsed_signup_std.isna()]

print("=== PARSING FAILURES (Syntax/Format) ===")
print(parsing_failures[['account_id', 'company', 'signup_ts']])


# Helper function to convert both Unix Epoch integers and ISO strings into datetime
def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    s_val = str(val).strip()
    if s_val.isdigit():
        return pd.to_datetime(int(s_val), unit='s')
    return pd.to_datetime(s_val, errors='coerce')

# Parse both columns into datetime objects for logical evaluation
df['signup_dt'] = df['signup_ts'].apply(parse_date)
df['last_active_dt'] = df['last_active'].apply(parse_date)


# 2. SURFACING VALIDATION PROBLEMS (Logical / Domain Failures)
# Validation Check A: Temporal paradox (Activity recorded before account creation)
paradox_failures = df[df['last_active_dt'] < df['signup_dt']]

# Validation Check B: Future sentinel dates (Activity recorded in distant future)
future_failures = df[df['last_active_dt'] > pd.Timestamp('2026-01-01')]

print("\n=== VALIDATION FAILURES (Logical Contradictions) ===")
print("1. Last Active Before Signup:")
print(paradox_failures[['account_id', 'company', 'signup_ts', 'last_active']])

print("\n2. Far Future / Sentinel Dates:")
print(future_failures[['account_id', 'company', 'signup_ts', 'last_active']])

=== PARSING FAILURES (Syntax/Format) ===
Empty DataFrame
Columns: [account_id, company, signup_ts]
Index: []

=== VALIDATION FAILURES (Logical Contradictions) ===
1. Last Active Before Signup:
  account_id company            signup_ts last_active
7     A-1007   Gamma  2023-06-30 00:00:00  2023-06-15

2. Far Future / Sentinel Dates:
   account_id company            signup_ts last_active
14     A-1013    Mike  2023-12-25 00:00:00  2099-01-01


##### Concept Explanation: Parsing vs. Validation
* Parsing (Syntactic Translation): Parsing is the process of converting an unformatted, raw string or raw numerical value into a structured data type (e.g., datetime64). A parsing problem occurs when the input string violates the expected grammar, syntax, or structure of the parser (e.g., passing a Unix timestamp integer to a string date parser).

* Validation (Semantic / Business Verification): Validation is the process of checking whether a successfully parsed, syntactically valid data point satisfies domain-specific business rules and physical logic. A validation problem occurs when a value is technically a valid date object, but represents an impossible or illegal state in the real world.

##### Specific Examples from SAAS.CSV
1. Parsing Problem Example
* Target: Row 3 (A-1003 - Cirrus) — signup_ts = '1675209600'

* Why it is a Parsing Failure: Standard Pandas string date parsers (pd.to_datetime(df['signup_ts'])) expect ISO strings (YYYY-MM-DD HH:MM:SS). Passing '1675209600' causes the string parser to crash with a DateParseError because it misinterprets the 10-digit integer string as a year. It requires explicit conversion from Unix epoch seconds into a datetime object.

2. Validation Problem Examples
* Target A: Row 7 (A-1007 - Gamma) — signup_ts = '2023-06-30', last_active = '2023-06-15'

* Why it is a Validation Failure: Both values parse without any errors into valid Timestamp objects. However, last_active < signup_ts fails business validation: a user cannot perform activity 15 days before their account was created.

* Target B: Row 14 (A-1013 - Mike) — last_active = '2099-01-01'

* Why it is a Validation Failure: '2099-01-01' parses perfectly into Timestamp('2099-01-01 00:00:00'). Syntactically, it is a valid date. Semantically, it fails validation because year 2099 is a hardcoded future sentinel value that invalidates user recency and tenure calculations.

## Section D: Judgement

#### Q9.The same null, two fates [8marks]
One row is missing a value in a label column; another is missing a value in a measure column. Identify both. Argue why the same action (say, dropping the row) could be correct for one and wrong for the other. Tie your argument to a specific downstream use of the data.

##### 1. Row Identification
* Missing Label Column (Row 15 - A-1014): Missing the company label (NaN), but has valid numerical measures (seats = 9.0, mrr = 810).

* Missing Measure Column (Row 8 - A-1008 - Helio): Missing the seats numerical measure (NaN), but has a valid company label (Helio) and mrr = 540.

##### 2. Argument Tied to Named Downstream Uses
###### Downstream Use 1: Executive Financial Reporting & Revenue Aggregation (Total MRR / ARR)
* Dropping Row 15 (Missing Label) is WRONG: Financial reporting calculates aggregate metrics across numerical measures (SUM(mrr)). Row 15 represents an active customer generating 810 in monthly recurring revenue. The missing company string is descriptive metadata; it has no bearing on financial sums. Dropping Row 15 using df.dropna() undercounts company revenue by 810/month (9,720 ARR). The correct action is to impute the label with 'Unknown'.

* Dropping Row 8 (Missing Measure) is ACCEPTABLE (if unimputed): In financial aggregation, if seats is a required weighting metric for per-seat revenue rollups and no business rules exist to recover missing measures, dropping unmeasurable rows prevents contaminating statistical distributions (or mrr / seats calculations) with NaN outputs.

###### Downstream Use 2: B2B CRM Integration & Sales Territory Routing
* Dropping Row 15 (Missing Label) is CORRECT: Lead routing scripts require a valid company name string to assign accounts to geographic territory sales reps or match against existing CRM account objects. An account lacking a company label cannot be matched, routed, or assigned in an automated sales pipeline.

* Dropping Row 8 (Missing Measure) is WRONG: In sales routing, account managers need the complete list of client names (Helio) and total contract values (mrr = 540) to manage client relationships. Dropping Helio removes a known paying customer from the sales rep's roster simply because their seat count field was unpopulated.

#### Q10.Cleaning is a function of the question [6marks]
Pick any two cleaning decisions you made earlier in this assessment. For each, describe a realistic goal under which your decision would reverse, you would do the opposite. Conclude in two sentences: what does this tell you about whether "clean" is a property of a dataset?
Submit: Two reversible decisions with their triggering goals, and your two-sentence conclusion

**Decision 1: Deduplicating Version-Drift Records (Row 1 vs. Row 12 — A-1002 Beta LLC)**
- Original Action: Dropped the older record (5 seats) to keep only the latest state (6 seats) for a single-row-per-account snapshot.

- Reversing Goal — Longitudinal Expansion & Cohort Analysis: If the downstream objective shifts to tracking Net Revenue Retention (NRR) or customer expansion over time, dropping the 5-seat record destroys historical event logs. To measure how accounts grow across billing cycles, you would preserve both records as distinct chronological events rather than collapsing them.

**Decision 2: Imputing Missing Measures via Baseline Rules (Row 8 — A-1008 Helio)**
- Original Action: Imputed seats = 6 by dividing the 540 MRR by the standard 90 Pro rate baseline.

- Reversing Goal — Sales Compliance & Discounting Audit: If the objective is auditing sales reps for unrecorded custom discounts or contract non-compliance, imputing seats using standard list prices creates circular logic that masks raw input errors. You would leave the measure as NaN or flag it as unverified to force manual reconciliation against signed Salesforce contracts.

##### Conclusion
This demonstrates that "clean" is not an intrinsic, objective property of a dataset, but a context-dependent state defined entirely by the downstream analytical goal. Data that is cleanly transformed for aggregate financial reporting can render evidence corrupted or destroyed for audit logging and time-series modelling.